# Parameters

In [ ]:
# Dummy parameters:
dummyVMax = 10000
dummyKm = 0.0001
dummyEc50 = 0.0001
dummyK = 0.5

h2o = 10000
proteinProductionVMax = dummyVMax
ribosomeKm = dummyKm

#NO3 to NO2
no3ReductionK = dummyK
no2OxidationK = dummyK
nrEc50 = dummyEc50

#NO2 to NO
no2ReductionK = dummyK
noOxidationK = dummyK
nirEc50 = dummyEc50

#NO to N2O
noReductionK = dummyK
n2oOxidationK = dummyK
norEc50 = dummyEc50

#N2O to N2
n2oReductionK = dummyK
n2OxidationK = dummyK
noszEc50 = dummyEc50

#Denitrification enzyme degradation
nrDegradationK = dummyK
nirDegradationK = dummyK
norDegradationK = dummyK
noszDegradationK = dummyK

#NR/NiR regulation
narxBindingK = dummyK
narxUnbindingK = dummyK
narxDegradationK = dummyK
narlPhosphorylationK = dummyK
narlpDephosphorylationK = dummyK
narlDegradationK = dummyK
narlpEc50 = dummyEc50


#NOR regulation
norrBindingK = dummyK
norrUnbindingK = dummyK
norrDegradationK = dummyK
norvwPhosphorylationK = dummyK
norvwpDephosphorylationK = dummyK
norvwDegradationK = dummyK
norvwpEc50 = dummyEc50

#NosZ regulation
nnrrBindingK = dummyK
nnrrUnbindingK = dummyK
nnrrDegradationK = dummyK
noNnrrEc50 = dummyEc50

# Nitrogen movement
nPumpVMax = dummyVMax
nPumpKm = dummyKm
n2DiffusionK = dummyK

# HPO4 to PO4
hpo4DehydrogenationK = dummyK
po4HydrogenationK = dummyK

#PO4 to Ca3PO4
precipitationK = dummyK

# Phosphorus movement
phosCellPumpVMax = dummyVMax
phosCellPumpKm = dummyKm
phosCompartmentPumpVMax = dummyVMax
phosCompartmentPumpKm = dummyKm

# Calcium movement
caDiffusionK = dummyK
caPumpVMax = dummyVMax
caCompartmentPumpKm = dummyKm

# Hydrogen movement
hDiffusionK = dummyK
hPumpVMax = dummyVMax
hPumpKm = dummyKm

# Transcription and mRNA degradation
transcriptionK = dummyK
mRNADegradationK = dummyK


# General Kinetic Equations

In [ ]:
def hill(s, km):
    return s/(km+s)

# Rate Equations

## Denitrofication

In [ ]:
def dNo3dt(_no3, _nr, _no2, _hCell, _narx, _no3Narx):
    return (nPumpVMax * hill(_no3, nPumpKm)
            - no3ReductionK * hill(_nr, nrEc50) * _no3 * _hCell ^ 2
            + no2OxidationK * hill(_nr, nrEc50) * _no2 * h2o
            - narxBindingK * _no3 * _narx
            + narxUnbindingK * _no3Narx)

def dNo2dt(_nr, _no3, _hCell, _no2, _nir, _no):
    return (no3ReductionK * hill(_nr, nrEc50) * _no3 *_hCell^2
            - no2OxidationK * hill(_nr, nrEc50) * _no2 * h2o
            - no2ReductionK * hill(_nir, nirEc50) * _no2 * _hCell^2
            + noOxidationK * hill(_nir, nirEc50) * _no * h2o)

def dNodt(_nir, _no2, _hCell, _no, _nor, _n2o, _norr, _noNorr, _nnrr, _noNnrr):
    return (no2ReductionK * hill(_nir, nirEc50) * _no2 * _hCell^2
            - noOxidationK * hill(_nir, nirEc50) * _no * h2o
            - noReductionK * hill(_nor, norEc50) * _no * _hCell^2
            + n2oOxidationK * hill(_nor, norEc50) * _n2o * h2o
            - norrBindingK * _no * _norr
            + norrUnbindingK * _noNorr
            - nnrrBindingK * _no * _nnrr
            + nnrrUnbindingK * _noNnrr)

def dN2odt(_nor, _no, _hCell, _n2o, _nosz, _n2, _norr, _noNorr):
    return(noReductionK * hill(_nor, norEc50) * _no * _hCell^2
           - n2oOxidationK * hill(_nor, norEc50) * _n2o * h2o
           - n2oReductionK * hill(_nosz, noszEc50) * _n2o * _hCell^2
           + n2OxidationK * hill(_nosz, noszEc50) * _n2 * h2o)

def dN2dt(_nosz, _n2o, _hCell, _n2):
    return (n2oReductionK * hill(_nosz, noszEc50) * _n2o * _hCell^2
            - n2OxidationK * hill(_nosz, noszEc50) * _n2 * h2o
            - n2DiffusionK * _n2)

def dNarxdt(_mRNA, _narx, _no3, _no3Narx):
    return (proteinProductionVMax * hill(_mRNA, ribosomeKm)
            - narxDegradationK * _narx
            - narxBindingK * _no3 * _narx
            + narxUnbindingK * _no3Narx)

def dNo3Narxdt(_no3, _narx, _no3Narx):
    return (narxBindingK * _no3 * _narx
            - narxUnbindingK * _no3Narx)

def dNarLdt(_mRNA, _narl, _no3Narx, _narlp):
    return (proteinProductionVMax * hill(_mRNA, ribosomeKm)
            - narlDegradationK * _narl
            - narlPhosphorylationK * _no3Narx * _narl
            + narlpDephosphorylationK * _narlp)

def dNarlpdt(_narl, _no3Narx, _narlp):
    return (narlPhosphorylationK * _no3Narx * _narl
            - narlpDephosphorylationK * _narlp)

def dNorrdt(_mRNA, _norr, _no, _noNorr):
    return (proteinProductionVMax * hill(_mRNA, ribosomeKm)
            - norrDegradationK * _norr
            - norrBindingK * _no * _norr
            + norrUnbindingK * _noNorr)

def dNoNorrdt(_no, _norr, _noNorr):
    return (norrBindingK * _no * _norr
            - norrUnbindingK * _noNorr)

def dNorvwdt(_mRNA, _norvw, _noNorr, _norvwp):
    return (proteinProductionVMax * hill(_mRNA, ribosomeKm)
            - norvwDegradationK * _norvw
            - norvwPhosphorylationK * _noNorr * _norvw
            + norvwpDephosphorylationK * _norvwp)

def dNorvwpdt(_norvw, _noNorr, _norvwp):
    return (norvwPhosphorylationK * _noNorr * _norvw
            - norvwpDephosphorylationK * _norvwp)

def dNnrrdt(_mRNA, _nnrr, _no, _noNnrr):
    return (proteinProductionVMax * hill(_mRNA, ribosomeKm)
            - nnrrDegradationK * _nnrr
            - nnrrBindingK * _no * _nnrr
            + nnrrUnbindingK * _noNnrr)

def dNoNnrrdt(_no, _nnrr, _noNnrr):
    return (nnrrBindingK * _no * _nnrr
            - nnrrUnbindingK * _noNnrr)

def dDenitroEnzymedt(_denitromRNA, _denitroEnzyme, denitroEnzymeDegradationK):
    return (proteinProductionVMax * hill(_denitromRNA, ribosomeKm)
            - denitroEnzymeDegradationK * _denitroEnzyme)

# def dNrdt(_nrmRNA, _nr):
#     return (proteinProductionVMax * hill(_nrmRNA, ribosomeKm)
#             - nrDegradationK * _nr)
# 
# def dNirdt(_nirmRNA, _nir):
#     return (proteinProductionVMax * hill(_nirmRNA, ribosomeKm)
#             - nirDegradationK * _nir)
# 
# def dNordt(_normRNA, _nor):
#     return (proteinProductionVMax * hill(_normRNA, ribosomeKm)
#             - norDegradationK * _nor)
# 
# def dNosZdt(_noszmRNA, _nosz):
#     return (proteinProductionVMax * hill(_noszmRNA, ribosomeKm)
#             - noszDegradationK * _nosz)


#HCell/dt (in these steps) -> in/out of cell, plus/minus all reduction steps

#NR manages no3 to no2, narx binds to no3 to phosphorylate narL which transcribes NR
#NiR manages no2 to no, same regulation
#NOR manages no to n2o, norR binds to no to phosphorylate norVW which transcribes NOR
#NosZ manages n2o to n2, no binds to NnrR which transcribes NosZ


## Phosphorus Precipitation

In [ ]:
def dHpo4Extracellulardt(_hpo4Extracellular):
    return -phosCellPumpVMax * hill(_hpo4Extracellular, phosCellPumpKm)

def dHpo4Celldt(_hpo4Extracellular, _hpo4Cell):
    return (phosCellPumpVMax * hill(_hpo4Extracellular, phosCellPumpKm)
            - phosCompartmentPumpVMax * hill(_hpo4Cell, phosCompartmentPumpKm))

def dHpo4Compartmentdt(_hpo4Cell, _hpo4Compartment, _po4, _hCompartment):
    return (phosCompartmentPumpVMax * hill(_hpo4Cell, phosCompartmentPumpKm)
            - hpo4DehydrogenationK * _hpo4Compartment
            + po4HydrogenationK * _po4 * _hCompartment)

def dPo4dt(_hpo4Compartment, _po4, _hCompartment, _caCompartment):
    return (hpo4DehydrogenationK * _hpo4Compartment
            - po4HydrogenationK * _po4 * _hCompartment
            - precipitationK * _caCompartment^3 * _po4)

def dCaCelldt(_caExtracellular, _caCell):
    return (caDiffusionK * (_caExtracellular - _caCell)
            - caPumpVMax * hill(_caCell, caCompartmentPumpKm))

def dCaCompartmentdt(_caCell, _caCompartment, _po4):
    return (caPumpVMax * hill(_caCell, caCompartmentPumpKm)
            - 3 * precipitationK * _caCompartment^3 * _po4)

def dCa3po4dt(_caCompartment, _po4):
    return precipitationK * _caCompartment^3 * _po4

# Calcium in/out cell => rate of diffusion of ca across membrane (estimate)
# Calcium in compartment => vmax * hill(ca, km)
# dehydrogenation equation = k*HPO4 (backwards is k*PO4*H)
# precipitation equation = k*ca*po4
# phosphorus pump => vmax * hill(phos, km)
# phosphorus pump into compartant = vmax * hill(phos, km)

## Hydrogen

In [ ]:
def dHExtracellulardt(_hExtracellular, _hCell):
    return -hDiffusionK * (_hExtracellular - _hCell)

def dHCelldt(_hExtracellular, _hCell, _hCompartment, _nr, _no3, _nir, _no2, _nor, _no, _nosz, _n2o, _n2):
    return (hDiffusionK * (_hExtracellular - _hCell) 
            + hPumpVMax * hill(_hCompartment, hPumpKm)
            - 2 * no3ReductionK * hill(_nr, nrEc50) * _no3 * _hCell^2
            - 2 * no2ReductionK * hill(_nir, nirEc50) * _no2 * _hCell^2
            - 2 * noReductionK * hill(_nor, norEc50) * _no * _hCell^2
            - 2 * n2oReductionK * hill(_nosz, noszEc50) * _n2o * _hCell^2
            + 2 * no2OxidationK * hill(_nr, nrEc50) * _no2 * h2o
            + 2 * noOxidationK * hill(_nir, nirEc50) * _no * h2o
            + 2 * n2oOxidationK * hill(_nor, norEc50) * _n2o * h2o
            + 2 * n2OxidationK * hill(_nosz, noszEc50) * _n2 * h2o)

def dHCompartmentdt(_hCell, _hCompartment, _hpo4Compartment, _po4):
    return (- hPumpVMax * hill(_hCompartment, hPumpKm)
            + hpo4DehydrogenationK * _hpo4Compartment
            - po4HydrogenationK * _po4 * _hCompartment)

# dH(compartment)/dt -> minus transport out of compartment, plus/minus from dehydrogenation of HPO4
# dH(cell)/dt -> plus transport out of compartment, plus/minus from all reduction steps,

## mRNA

In [ ]:
def dmRNAdt(_mRNA):
    return (transcriptionK
            - mRNADegradationK * _mRNA)

def dmRNAActivateddt(_mRNA, _activator, _activatorEc50):
    return (transcriptionK * hill(_activator, _activatorEc50)
            - mRNADegradationK * _mRNA)

# nr is activated by narL, nir is activated by narL, nor is activated by norVW, nosZ is activated by nnrr

## Total Rate Equation

In [ ]:
def dYdt(_t, _y):
    no3, no2, no, n2o, n2, nr, nir, nor, nosz, narx, no3Narx, narl, narlp, norr, noNorr, norvw, norvwp, nnrr, noNnrr, hpo4Extracellular, hpo4Cell, hpo4Compartment, po4, caExtracellular, caCell, caCompartment, ca3po4, hExtracellular, hCell, hCompartment, mRNA, nrmRNA, nirmRNA, normRNA, noszmRNA = _y
    return [dNo3dt(no3, nr, no2, hCell, narx, no3Narx),
            dNo2dt(nr, no3, hCell, no2, nir, no),
            dNodt(nir, no2, hCell, no, nor, n2o, norr, noNorr, nnrr, noNnrr),
            dN2odt(nor, no, hCell, n2o, nosz, n2, norr, noNorr),
            dN2dt(nosz, n2o, hCell, n2),
            dDenitroEnzymedt(nrmRNA, nr, nrDegradationK),
            dDenitroEnzymedt(nirmRNA, nir, nirDegradationK),
            dDenitroEnzymedt(normRNA, nor, norDegradationK),
            dDenitroEnzymedt(noszmRNA, nosz, noszDegradationK),
            dNarxdt(mRNA, narx, no3, no3Narx),
            dNo3Narxdt(no3, narx, no3Narx),
            dNarLdt(mRNA, narl, no3Narx, narlp),
            dNarlpdt(narl, no3Narx, narlp),
            dNorrdt(mRNA, norr, no, noNorr),
            dNoNorrdt(no, norr, noNorr),
            dNorvwdt(mRNA, norvw, noNorr, norvwp),
            dNorvwpdt(norvw, noNorr, norvwp),
            dNnrrdt(mRNA, nnrr, no, noNnrr),
            dNoNnrrdt(no, nnrr, noNnrr),
            dHpo4Extracellulardt(hpo4Extracellular),
            dHpo4Celldt(hpo4Extracellular, hpo4Cell),
            dHpo4Compartmentdt(hpo4Cell, hpo4Compartment, po4, hCompartment),
            dPo4dt(hpo4Compartment, po4, hCompartment, caCompartment),
            dCaCelldt(caExtracellular, caCell),
            dCaCompartmentdt(caCell, caCompartment, po4),
            dCa3po4dt(caCompartment, po4),
            dHExtracellulardt(hExtracellular, hCell),
            dHCelldt(hExtracellular, hCell, hCompartment, nr, no3, nir, no2, nor, no, nosz, n2o, n2),
            dHCompartmentdt(hCell, hCompartment, hpo4Compartment, po4),
            dmRNAdt(mRNA),
            dmRNAActivateddt(nrmRNA, narlp, narlpEc50),
            dmRNAActivateddt(nirmRNA, narlp, narlpEc50),
            dmRNAActivateddt(normRNA, norvwp, norvwpEc50),
            dmRNAActivateddt(noszmRNA, noNnrr, noNnrrEc50)]

# Initial Conditions

In [ ]:
dummyConcentration = 1

no3 = dummyConcentration
no2 = dummyConcentration
no = dummyConcentration
n2o = dummyConcentration
n2 = dummyConcentration
nr = dummyConcentration
nir = dummyConcentration
nor = dummyConcentration
nosz = dummyConcentration
narx = dummyConcentration
no3Narx = dummyConcentration
narl = dummyConcentration
narlp = dummyConcentration
norr = dummyConcentration
noNorr = dummyConcentration
norvw = dummyConcentration
norvwp = dummyConcentration
nnrr = dummyConcentration
noNnrr = dummyConcentration
hpo4Extracellular = dummyConcentration
hpo4Cell = dummyConcentration
hpo4Compartment = dummyConcentration
po4 = dummyConcentration
caExtracellular = dummyConcentration
caCell = dummyConcentration
caCompartment = dummyConcentration
ca3po4 = dummyConcentration
hExtracellular = dummyConcentration
hCell = dummyConcentration
hCompartment = dummyConcentration
mRNA = dummyConcentration
nrmRNA = dummyConcentration
nirmRNA = dummyConcentration
normRNA = dummyConcentration
noszmRNA = dummyConcentration